# EDA RetailRocket Ecommerce

Notebook de visualizacao:

Notebooks sao utilizados para exploracao e leitura dos resultados; artefatos finais serao gerados por scripts parametrizaveis.

Para atualizar os dados exibidos aqui, execute antes:

```powershell
uv --cache-dir .uv-cache run python scripts/run_retailrocket_eda.py --dataset-dir "D:/Dataset/archive"
```

- Ponto de atencao: DVC ainda nao configurado.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

METRICS_PATH = PROJECT_ROOT / "reports" / "eda" / "retailrocket_metrics.json"
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))


def int_br(value: int | None) -> str:
    if value is None:
        return "n/a"
    return f"{value:,}".replace(",", ".")


def pct(value: float) -> str:
    return f"{value:.6%}"


def bar_table(values: dict[str, int]) -> str:
    total = sum(values.values())
    max_value = max(values.values())
    rows = [_bar_row(key, value, total, max_value) for key, value in values.items()]
    return "<div class='bars'>" + "".join(rows) + "</div>"


def _bar_row(key: str, value: int, total: int, max_value: int) -> str:
    width = max(2, int((value / max_value) * 100))
    share = value / total if total else 0
    return f"""
    <div class='bar-row'>
      <div class='bar-label'>{key}</div>
      <div class='bar-track'><div class='bar-fill' style='width:{width}%'></div></div>
      <div class='bar-value'>{int_br(value)} ({share:.2%})</div>
    </div>
    """


display(HTML("""
<style>
.cards { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 12px; }
.card { border: 1px solid #d0d7de; border-radius: 8px; padding: 12px; }
.card .label { color: #57606a; font-size: 12px; text-transform: uppercase; }
.card .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
.bars { display: grid; gap: 10px; max-width: 900px; }
.bar-row { display: grid; grid-template-columns: 140px 1fr 160px; gap: 12px; align-items: center; }
.bar-track { background: #f1f3f5; border-radius: 999px; height: 14px; overflow: hidden; }
.bar-fill { background: #2f81f7; height: 100%; }
.bar-value { color: #57606a; font-variant-numeric: tabular-nums; }
table { border-collapse: collapse; width: 100%; }
th, td { border-bottom: 1px solid #d0d7de; padding: 8px; text-align: left; }
</style>
"""))


In [2]:
events = metrics["events"]
check = metrics["minimum_interactions"]

cards = [
    ("Eventos", int_br(events["row_count"])),
    ("Usuarios", int_br(events["unique_visitors"])),
    ("Itens", int_br(events["unique_items"])),
    ("Pares usuario-item", int_br(events["unique_user_item_pairs"])),
]

html_cards = "".join(
    f"<div class='card'><div class='label'>{label}</div><div class='value'>{value}</div></div>"
    for label, value in cards
)

display(Markdown("## Resumo executivo"))
display(HTML(f"<div class='cards'>{html_cards}</div>"))
display(Markdown(
    f"- Requisito minimo de {int_br(check['required_user_item_interactions'])} "
    f"interacoes: **{'atendido' if check['passes'] else 'nao atendido'}**.\n"
    f"- Esparsidade estimada da matriz usuario-item: **{pct(events['sparsity'])}**.\n"
    f"- Periodo dos eventos: **{events['date_range']['start']}** a "
    f"**{events['date_range']['end']}**."
))


## Resumo executivo

- Requisito minimo de 10.000 interacoes: **atendido**.
- Esparsidade estimada da matriz usuario-item: **99.999352%**.
- Periodo dos eventos: **2015-05-03** a **2015-09-18**.

In [3]:
event_counts = dict(
    sorted(events["event_counts"].items(), key=lambda item: item[1], reverse=True)
)

display(Markdown("## Distribuicao dos eventos"))
display(HTML(bar_table(event_counts)))


## Distribuicao dos eventos

In [4]:
monthly_counts = dict(events["monthly_counts"].items())

display(Markdown("## Distribuicao temporal"))
display(HTML(bar_table(monthly_counts)))
display(Markdown(
    "Split temporal sugerido: treino ate "
    f"**{events['temporal_split_cutoffs']['train_until']}**, validacao ate "
    f"**{events['temporal_split_cutoffs']['validation_until']}**."
))


## Distribuicao temporal

Split temporal sugerido: treino ate **2015-08-02**, validacao ate **2015-08-25**.

In [5]:
files = [
    metrics["events"],
    *metrics["item_properties"]["files"],
    metrics["category_tree"],
]

rows = "".join(
    "<tr>"
    f"<td>{file_data['file_name']}</td>"
    f"<td>{int_br(file_data['row_count'])}</td>"
    f"<td>{file_data['file_size_bytes'] / 1024 / 1024:.2f}</td>"
    f"<td>{', '.join(file_data['columns'])}</td>"
    "</tr>"
    for file_data in files
)

display(Markdown("## Inventario dos arquivos"))
display(HTML(
    "<table><thead><tr><th>Arquivo</th><th>Linhas</th><th>MiB</th>"
    "<th>Colunas</th></tr></thead><tbody>" + rows + "</tbody></table>"
))


## Inventario dos arquivos

Arquivo,Linhas,MiB,Colunas
events.csv,2.756.101,89.87,"timestamp, visitorid, event, itemid, transactionid"
item_properties_part1.csv,10.999.999,461.88,"timestamp, itemid, property, value"
item_properties_part2.csv,9.275.903,389.99,"timestamp, itemid, property, value"
category_tree.csv,1.669,0.01,"categoryid, parentid"


In [6]:
properties = metrics["item_properties"]
top_properties = {item["id"]: item["count"] for item in properties["top_properties"]}

display(Markdown("## Propriedades de itens"))
display(Markdown(
    f"- Linhas totais: **{int_br(properties['row_count'])}**.\n"
    f"- Itens com metadados: **{int_br(properties['unique_items'])}**.\n"
    f"- Propriedades unicas: **{int_br(properties['unique_properties'])}**.\n"
    f"- Itens com `categoryid`: **{int_br(properties['items_with_categoryid'])}**."
))
display(HTML(bar_table(top_properties)))


## Propriedades de itens

- Linhas totais: **20.275.902**.
- Itens com metadados: **417.053**.
- Propriedades unicas: **1.104**.
- Itens com `categoryid`: **417.053**.

In [7]:
recommendations = "\n".join(f"- {item}" for item in metrics["recommendations"])

display(Markdown("## Decisoes recomendadas"))
display(Markdown(recommendations))


## Decisoes recomendadas

- Usar split cronologico para evitar vazamento entre treino e avaliacao.
- Usar corte de treino em 2015-08-02 e validacao em 2015-08-25.
- Mapear eventos para pesos implicitos: view=1, addtocart=3, transaction=5.
- Usar categoryid como primeira feature de item.
- Tratar demais propriedades como metadados esparsos apos filtro de cardinalidade.
- Versionar CSVs brutos com DVC, sem commit direto no Git.